In [5]:
!pip install -q torch torchvision transformers pillow matplotlib tqdm

In [6]:
import os
import json
import math
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import CLIPProcessor, CLIPModel
from collections import defaultdict
import matplotlib.pyplot as plt

In [16]:
DATASET_DIR   = "/content/drive/MyDrive/eng521/Grasp Point Prediction/data/pixmo_subset_v3"
METADATA_PATH = os.path.join(DATASET_DIR, "metadata.json")
SPLITS_DIR    = os.path.join(DATASET_DIR, "splits")
TRAIN_PATH    = os.path.join(SPLITS_DIR, "train.json")
VAL_PATH      = os.path.join(SPLITS_DIR, "val.json")
TEST_PATH     = os.path.join(SPLITS_DIR, "test.json")
IMAGES_DIR    = os.path.join(DATASET_DIR, "images")

# v3 output folder
OUTPUT_DIR    = "/content/drive/MyDrive/eng521/Grasp Point Prediction/evaluation/Clip_v3_patch16"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

MODEL_NAME = "openai/clip-vit-base-patch16"

BATCH_SIZE = 16
EPOCHS     = 30
LR         = 1e-4
SUCCESS_THRESHOLD = 50

DEVICE: cpu


In [17]:
with open(TRAIN_PATH, "r") as f:
    train_data = json.load(f)
with open(VAL_PATH, "r") as f:
    val_data = json.load(f)
with open(TEST_PATH, "r") as f:
    test_data = json.load(f)

print("Train samples:", len(train_data))
print("Val samples:  ", len(val_data))
print("Test samples: ", len(test_data))
print("Example keys:", train_data[0].keys())

Train samples: 548
Val samples:   183
Test samples:  183
Example keys: dict_keys(['id', 'label', 'collection_method', 'count', 'points', 'image_path', 'image_url'])


In [18]:
def normalized_to_unit(pt):
    return float(pt["x"]) / 100.0, float(pt["y"]) / 100.0

def normalized_to_pixel(pt, width, height):
    x = float(pt["x"]) / 100.0 * width
    y = float(pt["y"]) / 100.0 * height
    return (x, y)

def unit_to_pixel(pred_xy, width, height):
    x = float(pred_xy[0]) * width
    y = float(pred_xy[1]) * height
    return (x, y)

def average_point(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return (sum(xs) / len(xs), sum(ys) / len(ys))

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def normalized_pixel_error(err, w, h):
    diagonal = math.sqrt(w**2 + h**2)
    return err / diagonal


def make_prompt_point(label):
    hints = {
        "backpack":    "Point to the strap of the backpack.",
        "bag":         "Point to the handle of the bag.",
        "calculator":  "Point to the body of the calculator.",
        "charger":     "Point to the cable of the charger.",
        "hairbrush":   "Point to the handle of the hairbrush.",
        "headphones":  "Point to the band of the headphones.",
        "highlighter": "Point to the body of the highlighter.",
        "jar":         "Point to the lid of the jar.",
        "kettle":      "Point to the handle of the kettle.",
        "keyboard":    "Point to the center of the keyboard.",
        "laptop":      "Point to the edge of the laptop.",
        "mouse":       "Point to the body of the mouse.",
        "notebook":    "Point to the spine of the notebook.",
        "onion":       "Point to the center of the onion.",
        "remote":      "Point to the body of the remote.",
        "scissor":     "Point to the handle of the scissor.",
        "stapler":     "Point to the top of the stapler.",
        "tape":        "Point to the side of the tape.",
        "toothbrush":  "Point to the handle of the toothbrush.",
        "umbrella":    "Point to the handle of the umbrella.",
        "wallet":      "Point to the body of the wallet.",
        "watch":       "Point to the strap of the watch.",
        "apple":       "Point to the center of the apple.",
        "book":        "Point to the spine of the book.",
        "bottle":      "Point to the neck of the bottle.",
        "bowl":        "Point to the rim of the bowl.",
        "cup":         "Point to the handle of the cup.",
        "fork":        "Point to the handle of the fork.",
        "keys":        "Point to the ring of the keys.",
        "knife":       "Point to the handle of the knife.",
        "marker":      "Point to the body of the marker.",
        "mug":         "Point to the handle of the mug.",
        "pen":         "Point to the body of the pen.",
        "phone":       "Point to the body of the phone.",
        "plate":       "Point to the edge of the plate.",
        "shoe":        "Point to the heel of the shoe.",
        "spoon":       "Point to the handle of the spoon.",
        "tomato":      "Point to the center of the tomato.",
        "tray":        "Point to the edge of the tray.",
        "basket":      "Point to the handle of the basket.",
        "pot":         "Point to the handle of the pot.",
        "pan":         "Point to the handle of the pan.",
        "sock":        "Point to the opening of the sock.",
        "glove":       "Point to the cuff of the glove.",
        "t-shirt":     "Point to the collar of the t-shirt.",
        "earbuds":     "Point to the body of the earbuds.",
        "egg":         "Point to the center of the egg.",
        "spatula":     "Point to the handle of the spatula.",
    }
    return hints.get(label.lower(), f"Point to the {label.lower()}.")


def make_prompt_grasp(label):
    hints = {
        "backpack":    "Grasp the strap of the backpack.",
        "bag":         "Grasp the handle of the bag.",
        "calculator":  "Grasp the body of the calculator.",
        "charger":     "Grasp the cable of the charger.",
        "hairbrush":   "Grasp the handle of the hairbrush.",
        "headphones":  "Grasp the band of the headphones.",
        "highlighter": "Grasp the body of the highlighter.",
        "jar":         "Grasp the lid of the jar.",
        "kettle":      "Grasp the handle of the kettle.",
        "keyboard":    "Grasp the center of the keyboard.",
        "laptop":      "Grasp the edge of the laptop.",
        "mouse":       "Grasp the body of the mouse.",
        "notebook":    "Grasp the spine of the notebook.",
        "onion":       "Grasp the center of the onion.",
        "remote":      "Grasp the body of the remote.",
        "scissor":     "Grasp the handle of the scissor.",
        "stapler":     "Grasp the top of the stapler.",
        "tape":        "Grasp the side of the tape.",
        "toothbrush":  "Grasp the handle of the toothbrush.",
        "umbrella":    "Grasp the handle of the umbrella.",
        "wallet":      "Grasp the body of the wallet.",
        "watch":       "Grasp the strap of the watch.",
        "apple":       "Grasp the center of the apple.",
        "book":        "Grasp the spine of the book.",
        "bottle":      "Grasp the neck of the bottle.",
        "bowl":        "Grasp the rim of the bowl.",
        "cup":         "Grasp the handle of the cup.",
        "fork":        "Grasp the handle of the fork.",
        "keys":        "Grasp the ring of the keys.",
        "knife":       "Grasp the handle of the knife.",
        "marker":      "Grasp the body of the marker.",
        "mug":         "Grasp the handle of the mug.",
        "pen":         "Grasp the body of the pen.",
        "phone":       "Grasp the body of the phone.",
        "plate":       "Grasp the edge of the plate.",
        "shoe":        "Grasp the heel of the shoe.",
        "spoon":       "Grasp the handle of the spoon.",
        "tomato":      "Grasp the center of the tomato.",
        "tray":        "Grasp the edge of the tray.",
        "basket":      "Grasp the handle of the basket.",
        "pot":         "Grasp the handle of the pot.",
        "pan":         "Grasp the handle of the pan.",
        "sock":        "Grasp the opening of the sock.",
        "glove":       "Grasp the cuff of the glove.",
        "t-shirt":     "Grasp the collar of the t-shirt.",
        "earbuds":     "Grasp the body of the earbuds.",
        "egg":         "Grasp the center of the egg.",
        "spatula":     "Grasp the handle of the spatula.",
    }
    return hints.get(label.lower(), f"Grasp the {label.lower()}.")


PROMPTS = {"point": make_prompt_point, "grasp": make_prompt_grasp}

In [19]:
from PIL import Image, UnidentifiedImageError

class PixmoClipDataset(Dataset):
    def __init__(self, data, images_dir):
        self.images_dir = images_dir
        self.data       = []
        self.skipped    = []

        for d in data:
            if len(d.get("points", [])) == 0:
                self.skipped.append(("no_points", d.get("image_path", "")))
                continue
            self.data.append(d)

        print("Clean dataset size:", len(self.data))
        print("Skipped samples:",    len(self.skipped))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample     = self.data[idx]
        image_name = os.path.basename(sample["image_path"])
        image_path = os.path.join(self.images_dir, image_name)
        image      = Image.open(image_path).convert("RGB")

        gt_points_unit = [normalized_to_unit(pt) for pt in sample["points"]]
        target_x = sum(p[0] for p in gt_points_unit) / len(gt_points_unit)
        target_y = sum(p[1] for p in gt_points_unit) / len(gt_points_unit)

        return {
            "image":      image,
            "text":       make_prompt_point(sample["label"]),
            "target":     torch.tensor([target_x, target_y], dtype=torch.float32),
            "label":      sample["label"],
            "raw_sample": sample,
            "image_path": image_path,
            "orig_size":  image.size
        }

In [20]:
train_dataset = PixmoClipDataset(train_data, IMAGES_DIR)
val_dataset   = PixmoClipDataset(val_data,   IMAGES_DIR)
test_dataset  = PixmoClipDataset(test_data,  IMAGES_DIR)

print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))

Clean dataset size: 548
Skipped samples: 0
Clean dataset size: 183
Skipped samples: 0
Clean dataset size: 183
Skipped samples: 0
Train: 548
Val:   183
Test:  183


In [22]:
processor = CLIPProcessor.from_pretrained(MODEL_NAME)

def collate_fn(batch):
    images  = [item["image"]  for item in batch]
    texts   = [item["text"]   for item in batch]
    targets = torch.stack([item["target"] for item in batch])

    enc = processor(
        text=texts, images=images,
        return_tensors="pt", padding=True
    )
    return {
        "pixel_values":   enc["pixel_values"],
        "input_ids":      enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "targets":        targets,
        "meta":           batch
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [23]:
# ── v3: Heatmap-based model ───────────────────────────────────────────────────
# Instead of regressing (x,y) directly, score each patch and do a
# softmax-weighted sum over the 7x7 spatial grid → predicted point.
# This forces spatial grounding: the model must learn WHICH patch to attend to.

class CLIPHeatmapRegressor(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

        # Freeze all CLIP parameters
        for param in self.clip.parameters():
            param.requires_grad = False

        feat_dim  = self.clip.config.projection_dim             # 512
        patch_dim = self.clip.config.vision_config.hidden_size  # 768

        # Text-guided cross-attention over patches
        self.text_proj  = nn.Linear(feat_dim, patch_dim)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=patch_dim, num_heads=1, batch_first=True
        )
        self.attn_norm  = nn.LayerNorm(patch_dim)

        # Score each patch → softmax → weighted sum of patch coordinates
        self.patch_scorer = nn.Sequential(
            nn.Linear(patch_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )

    def forward(self, pixel_values, input_ids, attention_mask):
        outputs = self.clip(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
            output_hidden_states=True
        )
        text_feats  = outputs.text_embeds                                        # (B, 512)
        patch_feats = outputs.vision_model_output.last_hidden_state[:, 1:, :]   # (B, 49, 768)

        B, N, D = patch_feats.shape   # N=49 for ViT-B/32
        grid_size = int(N ** 0.5)     # 7

        # Cross-attention: text queries over patch tokens
        query    = self.text_proj(text_feats).unsqueeze(1)          # (B, 1, 768)
        attn_out, _ = self.cross_attn(query, patch_feats, patch_feats)
        spatial  = self.attn_norm(attn_out.squeeze(1))              # (B, 768)

        # Gate patch features with spatial context
        gated_patches = patch_feats + spatial.unsqueeze(1)          # (B, 49, 768)

        # Score each patch
        scores  = self.patch_scorer(gated_patches).squeeze(-1)      # (B, 49)
        weights = torch.softmax(scores, dim=-1)                     # (B, 49)

        # Build 2D coordinate grid matching patch layout
        xs = torch.linspace(0, 1, grid_size, device=pixel_values.device)
        ys = torch.linspace(0, 1, grid_size, device=pixel_values.device)
        grid_y, grid_x = torch.meshgrid(ys, xs, indexing="ij")
        coords = torch.stack(
            [grid_x.flatten(), grid_y.flatten()], dim=-1
        ).unsqueeze(0)                                              # (1, 49, 2)

        # Weighted sum → predicted (x, y) in [0,1]
        pred = (weights.unsqueeze(-1) * coords).sum(dim=1)         # (B, 2)
        return pred

    def get_heatmap(self, pixel_values, input_ids, attention_mask):
        """Return patch weights as 7x7 heatmap for visualization."""
        with torch.no_grad():
            outputs = self.clip(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True,
                output_hidden_states=True
            )
            text_feats  = outputs.text_embeds
            patch_feats = outputs.vision_model_output.last_hidden_state[:, 1:, :]

            B, N, D   = patch_feats.shape
            grid_size = int(N ** 0.5)

            query    = self.text_proj(text_feats).unsqueeze(1)
            attn_out, _ = self.cross_attn(query, patch_feats, patch_feats)
            spatial  = self.attn_norm(attn_out.squeeze(1))
            gated    = patch_feats + spatial.unsqueeze(1)
            scores   = self.patch_scorer(gated).squeeze(-1)
            weights  = torch.softmax(scores, dim=-1)

        return weights.squeeze().cpu().numpy().reshape(grid_size, grid_size)

In [24]:
def train_one_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader, desc="train", leave=False):
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets        = batch["targets"].to(device)

        optimizer.zero_grad()
        preds = model(pixel_values, input_ids, attention_mask)
        loss  = F.smooth_l1_loss(preds, targets, beta=0.05)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item() * targets.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_one_epoch(model, loader, device):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(loader, desc="val", leave=False):
        pixel_values   = batch["pixel_values"].to(device)
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets        = batch["targets"].to(device)

        preds = model(pixel_values, input_ids, attention_mask)
        loss  = F.smooth_l1_loss(preds, targets, beta=0.05)
        total_loss += loss.item() * targets.size(0)
    return total_loss / len(loader.dataset)

In [25]:
from transformers import get_cosine_schedule_with_warmup

model = CLIPHeatmapRegressor().to(DEVICE)

# Sanity check
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

optimizer = torch.optim.AdamW([
    {"params": model.text_proj.parameters(),    "lr": LR},
    {"params": model.cross_attn.parameters(),   "lr": LR},
    {"params": model.attn_norm.parameters(),    "lr": LR},
    {"params": model.patch_scorer.parameters(), "lr": LR},
], weight_decay=1e-2)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader) * 2,
    num_training_steps=len(train_loader) * EPOCHS
)

best_val_loss    = float("inf")
patience         = 8
patience_counter = 0
MODEL_SAVE_PATH  = "/content/drive/MyDrive/clip_grasp_v3_best.pt"

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE, scheduler)
    val_loss   = eval_one_epoch(model, val_loader, DEVICE)
    print(f"Epoch {epoch+1}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  ✅ New best: {best_val_loss:.4f} — saved")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(MODEL_SAVE_PATH))
print("Loaded best model weights for evaluation")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch16
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Trainable params: 2,807,169 / 152,427,906 (1.8%)


Epoch 1/30 | train=0.1807 | val=0.1829
  ✅ New best: 0.1829 — saved


Epoch 2/30 | train=0.1789 | val=0.1816
  ✅ New best: 0.1816 — saved


...

Epoch 25/30 | train=0.1223 | val=0.1554
  ⏳ No improvement (4/8)


Epoch 26/30 | train=0.1219 | val=0.1551
  ⏳ No improvement (5/8)


Epoch 27/30 | train=0.1217 | val=0.1551
  ⏳ No improvement (6/8)


Epoch 28/30 | train=0.1215 | val=0.1550
  ⏳ No improvement (7/8)


Epoch 29/30 | train=0.1214 | val=0.1551
  ⏳ No improvement (8/8)
Early stopping at epoch 29
Loaded best model weights for evaluation


In [26]:
# ── EVALUATION FUNCTIONS — GD-matching output format ─────────────────────────

@torch.no_grad()
def run_prompt(model, data, images_dir, device, prompt_name, prompt_fn, split):
    model.eval()
    results = []

    for i, sample in enumerate(tqdm(data, desc=f"  {prompt_name}")):
        image_name = os.path.basename(sample["image_path"])
        image_path = os.path.join(images_dir, image_name)
        label      = sample["label"]

        image = Image.open(image_path).convert("RGB")
        w, h  = image.size
        text  = prompt_fn(label)

        enc = processor(
            text=[text], images=[image],
            return_tensors="pt", padding=True
        )
        pixel_values   = enc["pixel_values"].to(device)
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        pred    = model(pixel_values, input_ids, attention_mask).cpu().numpy()[0]
        pred_px = unit_to_pixel(pred, w, h)

        gt_points_px = [normalized_to_pixel(pt, w, h) for pt in sample["points"]]
        gt_avg   = average_point(gt_points_px)
        err      = euclidean_distance(pred_px, gt_avg)
        norm_err = normalized_pixel_error(err, w, h)

        results.append({
            "id":         sample.get("id", i),
            "split":      split,
            "image_path": image_path,
            "filename":   image_name,
            "label":      label,
            "prompt":     text,
            "pred_x":     round(float(pred_px[0]), 6),
            "pred_y":     round(float(pred_px[1]), 6),
            "pred_point": [round(float(pred_px[0]), 6), round(float(pred_px[1]), 6)],
            "gt_x":       round(float(gt_avg[0]), 6),
            "gt_y":       round(float(gt_avg[1]), 6),
            "gt_point":   [round(float(gt_avg[0]), 6), round(float(gt_avg[1]), 6)],
            "pixel_error":            round(float(err), 6),
            "normalized_pixel_error": round(float(norm_err), 6),
            "error":            round(float(err), 6),
            "normalized_error": round(float(norm_err), 6),
            "success":    int(err < SUCCESS_THRESHOLD),
            "status":     "ok",
            "score":      None,
            "best_label": None,
        })

    return results


def _prompt_summary(results, prompt_name):
    valid = [r for r in results if r["pixel_error"] is not None]
    n, nv = len(results), len(valid)
    return {
        "template":              prompt_name,
        "mean_error":            float(np.mean([r["pixel_error"]            for r in valid])) if valid else None,
        "mean_normalized_error": float(np.mean([r["normalized_pixel_error"] for r in valid])) if valid else None,
        "success_rate":          float(np.mean([r["success"]                for r in valid])) if valid else None,
        "detection_rate":        nv / n if n > 0 else 0.0,
        "num_samples":           n,
        "num_valid":             nv,
    }


def evaluate_all_prompts(model, data, images_dir, device, split, output_dir):
    prompt_eval    = []
    prompt_summary = []

    for prompt_name, prompt_fn in PROMPTS.items():
        print(f"\nPrompt: '{prompt_name}'")
        results = run_prompt(model, data, images_dir, device, prompt_name, prompt_fn, split)
        summ    = _prompt_summary(results, prompt_name)

        details = [{
            "id":               r["id"],
            "label":            r["label"],
            "prompt":           r["prompt"],
            "filename":         r["filename"],
            "gt_point":         r["gt_point"],
            "pred_point":       r["pred_point"],
            "error":            r["error"],
            "normalized_error": r["normalized_error"],
            "score":            r["score"],
            "best_label":       r["best_label"],
            "status":           r["status"],
        } for r in results]

        prompt_eval.append({**summ, "details": details})
        prompt_summary.append(summ)
        print(f"  mean_error={summ['mean_error']:.4f} | success_rate={summ['success_rate']:.4f}")

    prompt_summary_sorted = sorted(prompt_summary, key=lambda x: x["mean_error"])
    best_prompt_name      = prompt_summary_sorted[0]["template"]
    print(f"\n✅ Best prompt on '{split}': '{best_prompt_name}'")

    prefix = f"clip_{split}"
    with open(os.path.join(output_dir, f"{prefix}_prompt_eval.json"),    "w") as f:
        json.dump(prompt_eval,           f, indent=2)
    with open(os.path.join(output_dir, f"{prefix}_prompt_summary.json"), "w") as f:
        json.dump(prompt_summary_sorted, f, indent=2)
    print(f"Saved: {prefix}_prompt_eval.json, {prefix}_prompt_summary.json")

    return prompt_eval, prompt_summary_sorted, best_prompt_name


def evaluate_with_best_prompt(model, data, images_dir, device, split, best_prompt_name, output_dir):
    prompt_fn = PROMPTS[best_prompt_name]
    print(f"\nEvaluating '{split}' with best prompt: '{best_prompt_name}'")
    results = run_prompt(model, data, images_dir, device, best_prompt_name, prompt_fn, split)

    # per_image
    per_image = [{
        "split":                  r["split"],
        "image_path":             r["image_path"],
        "label":                  r["label"],
        "prompt":                 r["prompt"],
        "pred_x":                 r["pred_x"],
        "pred_y":                 r["pred_y"],
        "gt_x":                   r["gt_x"],
        "gt_y":                   r["gt_y"],
        "pixel_error":            r["pixel_error"],
        "normalized_pixel_error": r["normalized_pixel_error"],
        "success":                r["success"],
        "status":                 r["status"],
        "score":                  r["score"],
        "best_label":             r["best_label"],
        "best_prompt_from_val":   best_prompt_name,
        "prompt_sensitivity":     None,
        "prompt_consistency":     None,
    } for r in results]

    # per_category
    cat_groups = defaultdict(list)
    for r in results:
        cat_groups[r["label"]].append(r)

    per_category = []
    for label, recs in sorted(cat_groups.items(),
                               key=lambda x: np.mean([r["pixel_error"] for r in x[1]])):
        valid = [r for r in recs if r["pixel_error"] is not None]
        n, nv = len(recs), len(valid)
        per_category.append({
            "label":                 label,
            "num_samples":           n,
            "num_valid":             nv,
            "mean_error":            float(np.mean([r["pixel_error"]            for r in valid])) if valid else None,
            "mean_normalized_error": float(np.mean([r["normalized_pixel_error"] for r in valid])) if valid else None,
            "success_rate":          float(np.mean([r["success"]                for r in valid])) if valid else None,
            "detection_rate":        nv / n if n > 0 else 0.0,
        })

    # final_summary
    valid_all     = [r for r in results if r["pixel_error"] is not None]
    final_summary = {
        "split":              split,
        "best_prompt":        best_prompt_name,
        "success_rate":       float(np.mean([r["success"]      for r in valid_all])) if valid_all else None,
        "per_prompt_error":   {best_prompt_name: float(np.mean([r["pixel_error"] for r in valid_all]))} if valid_all else {},
        "per_category_error": {pc["label"]: pc["mean_error"] for pc in per_category},
        "prompt_sensitivity": None,
        "prompt_consistency": None,
    }

    prefix = f"clip_{split}"
    with open(os.path.join(output_dir, f"{prefix}_final_per_image.json"),    "w") as f:
        json.dump(per_image,     f, indent=2)
    with open(os.path.join(output_dir, f"{prefix}_final_per_category.json"), "w") as f:
        json.dump(per_category,  f, indent=2)
    with open(os.path.join(output_dir, f"{prefix}_final_summary.json"),      "w") as f:
        json.dump(final_summary, f, indent=2)
    print(f"Saved: {prefix}_final_per_image.json, _per_category.json, _final_summary.json")

    return per_image, per_category, final_summary


@torch.no_grad()
def evaluate_per_image_best_prompt(model, data, images_dir, device, split, output_dir):
    """Oracle upper bound: pick best prompt per image."""
    model.eval()
    results = []

    for i, sample in enumerate(tqdm(data, desc=" per-image best")):
        image_name = os.path.basename(sample["image_path"])
        image_path = os.path.join(images_dir, image_name)
        label      = sample["label"]

        image = Image.open(image_path).convert("RGB")
        w, h  = image.size

        gt_points_px = [normalized_to_pixel(pt, w, h) for pt in sample["points"]]
        gt_avg       = average_point(gt_points_px)

        best_err    = float("inf")
        best_result = None

        for prompt_name, prompt_fn in PROMPTS.items():
            text = prompt_fn(label)
            enc  = processor(
                text=[text], images=[image],
                return_tensors="pt", padding=True
            )
            pred    = model(
                enc["pixel_values"].to(device),
                enc["input_ids"].to(device),
                enc["attention_mask"].to(device)
            ).cpu().numpy()[0]
            pred_px  = unit_to_pixel(pred, w, h)
            err      = euclidean_distance(pred_px, gt_avg)
            norm_err = normalized_pixel_error(err, w, h)

            if err < best_err:
                best_err    = err
                best_result = {
                    "id":               sample.get("id", i),
                    "split":            split,
                    "image_path":       image_path,
                    "filename":         image_name,
                    "label":            label,
                    "prompt":           text,
                    "best_prompt_name": prompt_name,
                    "pred_x":           round(float(pred_px[0]), 6),
                    "pred_y":           round(float(pred_px[1]), 6),
                    "pred_point":       [round(float(pred_px[0]), 6), round(float(pred_px[1]), 6)],
                    "gt_x":             round(float(gt_avg[0]), 6),
                    "gt_y":             round(float(gt_avg[1]), 6),
                    "gt_point":         [round(float(gt_avg[0]), 6), round(float(gt_avg[1]), 6)],
                    "pixel_error":            round(float(err), 6),
                    "normalized_pixel_error": round(float(norm_err), 6),
                    "success":          int(err < SUCCESS_THRESHOLD),
                    "status":           "ok",
                }

        results.append(best_result)

    # per_category
    cat_groups = defaultdict(list)
    for r in results:
        cat_groups[r["label"]].append(r)

    per_category = []
    for label, recs in sorted(cat_groups.items(),
                               key=lambda x: np.mean([r["pixel_error"] for r in x[1]])):
        valid = [r for r in recs if r["pixel_error"] is not None]
        n, nv = len(recs), len(valid)
        per_category.append({
            "label":        label,
            "num_samples":  n,
            "num_valid":    nv,
            "mean_error":   float(np.mean([r["pixel_error"]            for r in valid])) if valid else None,
            "success_rate": float(np.mean([r["success"]                for r in valid])) if valid else None,
            "detection_rate": nv / n if n > 0 else 0.0,
        })

    valid_all     = [r for r in results if r["pixel_error"] is not None]
    final_summary = {
        "split":        split,
        "best_prompt":  "per_image_best",
        "success_rate": float(np.mean([r["success"] for r in valid_all])) if valid_all else None,
        "mean_error":   float(np.mean([r["pixel_error"] for r in valid_all])) if valid_all else None,
        "per_category_error": {pc["label"]: pc["mean_error"] for pc in per_category},
    }

    prefix = f"clip_{split}_per_image_best"
    with open(os.path.join(output_dir, f"{prefix}_per_image.json"),    "w") as f:
        json.dump(results,       f, indent=2)
    with open(os.path.join(output_dir, f"{prefix}_per_category.json"), "w") as f:
        json.dump(per_category,  f, indent=2)
    with open(os.path.join(output_dir, f"{prefix}_summary.json"),      "w") as f:
        json.dump(final_summary, f, indent=2)
    print(f"Saved: {prefix}_*.json")

    return results, per_category, final_summary

In [27]:
# ── 1. Find best prompt on val ────────────────────────────────────────────────
val_prompt_eval, val_prompt_summary, BEST_PROMPT = evaluate_all_prompts(
    model, val_data, IMAGES_DIR, DEVICE, split="val", output_dir=OUTPUT_DIR
)

# ── 2. Final val evaluation with best prompt ──────────────────────────────────
val_per_image, val_per_category, val_final_summary = evaluate_with_best_prompt(
    model, val_data, IMAGES_DIR, DEVICE,
    split="val", best_prompt_name=BEST_PROMPT, output_dir=OUTPUT_DIR
)

# ── 3. Final test evaluation with best prompt ─────────────────────────────────
test_per_image, test_per_category, test_final_summary = evaluate_with_best_prompt(
    model, test_data, IMAGES_DIR, DEVICE,
    split="test", best_prompt_name=BEST_PROMPT, output_dir=OUTPUT_DIR
)

# ── 4. Per-image oracle best prompt (test) ────────────────────────────────────
test_pib, test_pib_cat, test_pib_summary = evaluate_per_image_best_prompt(
    model, test_data, IMAGES_DIR, DEVICE, split="test", output_dir=OUTPUT_DIR
)

# ── Print final summary ───────────────────────────────────────────────────────
print("\n========== FINAL RESULTS ==========")
print(f"Best prompt (from val)          : '{BEST_PROMPT}'")
print(f"Val  success rate               : {val_final_summary['success_rate']*100:.2f}%")
print(f"Test success rate (best prompt) : {test_final_summary['success_rate']*100:.2f}%")
print(f"Test success rate (per-img best): {test_pib_summary['success_rate']*100:.2f}%")

print("\n--- Val Prompt Comparison ---")
for ps in val_prompt_summary:
    print(f"  {ps['template']:<10}: mean_error={ps['mean_error']:.2f}px | "
          f"SR={ps['success_rate']*100:.2f}% | "
          f"detection_rate={ps['detection_rate']:.4f}")

print("\n--- Test Per-Category Error (sorted best → worst) ---")
for pc in test_per_category:
    sr = pc['success_rate'] if pc['success_rate'] is not None else 0.0
    print(f"  {pc['label']:<15}: mean={pc['mean_error']:.2f}px | "
          f"SR={sr*100:.1f}% | n={pc['num_samples']}")

errors  = [r["pixel_error"] for r in test_per_image]
success = [r["success"]     for r in test_per_image]
print("\n--- Test Pixel Error Stats (best prompt) ---")
print(f"  Mean   : {np.mean(errors):.2f} px")
print(f"  Median : {np.median(errors):.2f} px")
print(f"  Std    : {np.std(errors):.2f} px")
print(f"  Min    : {np.min(errors):.2f} px")
print(f"  Max    : {np.max(errors):.2f} px")
print(f"  Success rate: {np.mean(success)*100:.2f}% ({sum(success)}/{len(success)})")

pib_errors  = [r["pixel_error"] for r in test_pib]
pib_success = [r["success"]     for r in test_pib]
print("\n--- Test Pixel Error Stats (per-image best — oracle) ---")
print(f"  Mean   : {np.mean(pib_errors):.2f} px")
print(f"  Median : {np.median(pib_errors):.2f} px")
print(f"  Std    : {np.std(pib_errors):.2f} px")
print(f"  Min    : {np.min(pib_errors):.2f} px")
print(f"  Max    : {np.max(pib_errors):.2f} px")
print(f"  Success rate: {np.mean(pib_success)*100:.2f}% ({sum(pib_success)}/{len(pib_success)})")
print("====================================")


Prompt: 'point'


  point: 100%|██████████| 183/183 [02:41<00:00,  1.13it/s]


  mean_error=279.4416 | success_rate=0.1093

Prompt: 'grasp'


  grasp: 100%|██████████| 183/183 [02:37<00:00,  1.16it/s]


  mean_error=279.7167 | success_rate=0.1093

✅ Best prompt on 'val': 'point'
Saved: clip_val_prompt_eval.json, clip_val_prompt_summary.json

Evaluating 'val' with best prompt: 'point'


...

Saved: clip_val_final_per_image.json, _per_category.json, _final_summary.json

Evaluating 'test' with best prompt: 'point'


  point: 100%|██████████| 183/183 [03:38<00:00,  1.20s/it]


Saved: clip_test_final_per_image.json, _per_category.json, _final_summary.json


 per-image best: 100%|██████████| 183/183 [05:15<00:00,  1.72s/it]

Saved: clip_test_per_image_best_*.json

========== FINAL RESULTS ==========
Best prompt (from val)          : 'point'
Val  success rate               : 10.93%
Test success rate (best prompt) : 6.01%
Test success rate (per-img best): 6.01%

--- Val Prompt Comparison ---
  point     : mean_error=279.44px | SR=10.93% | detection_rate=1.0000
  grasp     : mean_error=279.72px | SR=10.93% | detection_rate=1.0000

--- Test Per-Category Error (sorted best → worst) ---
  watch          : mean=66.95px | SR=50.0% | n=4
  spatula        : mean=77.67px | SR=0.0% | n=4
  umbrella       : mean=98.08px | SR=25.0% | n=4
  backpack       : mean=108.01px | SR=0.0% | n=4
  book           : mean=114.05px | SR=0.0% | n=4
  onion          : mean=144.48px | SR=0.0% | n=4
  scissor        : mean=146.57px | SR=0.0% | n=4
  mouse          : mean=149.48px | SR=0.0% | n=4
  keys           : mean=150.85px | SR=0.0% | n=4
  notebook       : mean=157.40px | SR=25.0% | n=4
  tomato         : mean=159.80px | SR=0.0% | 

In [28]:
# ── Visualization helpers ─────────────────────────────────────────────────────

def show_prediction(per_image_results, idx=0):
    row = per_image_results[idx]
    img = Image.open(row["image_path"]).convert("RGB")

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.scatter(row["gt_x"],   row["gt_y"],  s=150, marker="o", color="green",
                label="GT")
    plt.scatter(row["pred_x"], row["pred_y"], s=150, marker="x", color="red",
                label=f"Pred (err={row['pixel_error']:.1f}px)")
    status = "✓" if row["success"] == 1 else "✗"
    plt.title(f"{row['label']} | {status} | err={row['pixel_error']:.1f}px")
    plt.legend()
    plt.axis("off")
    plt.show()


def visualize_heatmap(model, sample, images_dir, device):
    """Show patch heatmap, blended attention, and prediction vs GT."""
    image_path = os.path.join(images_dir, os.path.basename(sample["image_path"]))
    image = Image.open(image_path).convert("RGB")
    w, h  = image.size
    text  = PROMPTS[BEST_PROMPT](sample["label"])

    enc = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    pixel_values   = enc["pixel_values"].to(device)
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    # Heatmap
    heatmap = model.get_heatmap(pixel_values, input_ids, attention_mask)

    # Prediction
    model.eval()
    with torch.no_grad():
        pred = model(pixel_values, input_ids, attention_mask).cpu().numpy()[0]
    pred_px = unit_to_pixel(pred, w, h)
    gt_px   = average_point([normalized_to_pixel(pt, w, h) for pt in sample["points"]])
    err     = euclidean_distance(pred_px, gt_px)

    # Overlay
    heat     = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    heat_img = Image.fromarray(
        (plt.cm.jet(heat)[:, :, :3] * 255).astype(np.uint8)
    ).resize((w, h), Image.BILINEAR)
    blended  = Image.blend(image, heat_img, alpha=0.5)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(blended)
    axes[1].scatter(*pred_px, s=150, c="red",   marker="x", label="Pred", zorder=5)
    axes[1].scatter(*gt_px,   s=150, c="green", marker="o", label="GT",   zorder=5)
    axes[1].set_title(f"Heatmap + Pred\nerr={err:.1f}px | '{text}'")
    axes[1].legend()
    axes[1].axis("off")

    axes[2].imshow(heatmap, cmap="jet", interpolation="bilinear")
    axes[2].set_title(f"Raw Heatmap ({heatmap.shape[0]}×{heatmap.shape[1]})")
    axes[2].axis("off")

    status = "✓" if err < SUCCESS_THRESHOLD else "✗"
    plt.suptitle(f"{sample['label']} | {status}", fontsize=13)
    plt.tight_layout()
    plt.show()


def visualize_attention_grid(model, data, images_dir, device, n=6, mode="random"):
    import random as rnd
    if mode == "random":
        samples = rnd.sample(data, min(n, len(data)))
    elif mode == "worst":
        sorted_r = sorted(test_per_image, key=lambda r: r["pixel_error"], reverse=True)[:n]
        path_map = {os.path.join(images_dir, os.path.basename(s["image_path"])): s for s in data}
        samples  = [path_map[r["image_path"]] for r in sorted_r if r["image_path"] in path_map]
    elif mode == "best":
        sorted_r = sorted(test_per_image, key=lambda r: r["pixel_error"])[:n]
        path_map = {os.path.join(images_dir, os.path.basename(s["image_path"])): s for s in data}
        samples  = [path_map[r["image_path"]] for r in sorted_r if r["image_path"] in path_map]
    for s in samples:
        visualize_heatmap(model, s, images_dir, device)

In [29]:
# Sample predictions
show_prediction(test_per_image, idx=0)
show_prediction(test_per_image, idx=5)

# Heatmap visualizations
print("=== BEST PREDICTIONS ===")
visualize_attention_grid(model, test_data, IMAGES_DIR, DEVICE, n=3, mode="best")

print("=== WORST PREDICTIONS ===")
visualize_attention_grid(model, test_data, IMAGES_DIR, DEVICE, n=3, mode="worst")

print("=== RANDOM SAMPLES ===")
visualize_attention_grid(model, test_data, IMAGES_DIR, DEVICE, n=4, mode="random")

Output hidden; open in https://colab.research.google.com to view.

In [30]:
torch.save(model.state_dict(), "/content/drive/MyDrive/clip_grasp_v3_best.pt")
print("Model saved.")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
print("  clip_val_prompt_eval.json")
print("  clip_val_prompt_summary.json")
print("  clip_val_final_per_image.json")
print("  clip_val_final_per_category.json")
print("  clip_val_final_summary.json")
print("  clip_test_final_per_image.json")
print("  clip_test_final_per_category.json")
print("  clip_test_final_summary.json")
print("  clip_test_per_image_best_per_image.json")
print("  clip_test_per_image_best_per_category.json")
print("  clip_test_per_image_best_summary.json")

Model saved.

All outputs saved to: /content/drive/MyDrive/eng521/Grasp Point Prediction/evaluation/Clip_v3_patch16
  clip_val_prompt_eval.json
  clip_val_prompt_summary.json
  clip_val_final_per_image.json
  clip_val_final_per_category.json
  clip_val_final_summary.json
  clip_test_final_per_image.json
  clip_test_final_per_category.json
  clip_test_final_summary.json
  clip_test_per_image_best_per_image.json
  clip_test_per_image_best_per_category.json
  clip_test_per_image_best_summary.json


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
OUTPUT_DIR    = "/content/drive/MyDrive/eng521"
fig, ax = plt.subplots(figsize=(9, 5))

clip_models  = ["Baseline\n(mean-pool)", "Cross-Attention", "Heatmap (v3)"]
clip_sr      = [1.64, 2.73, 6.01]
clip_err     = [323, 308, 280]
x = np.arange(len(clip_models))
width = 0.35

bars1 = ax.bar(x - width/2, clip_sr,  width, label="Success Rate (%)",  color="#5cb85c", alpha=0.85)
ax2   = ax.twinx()
bars2 = ax2.bar(x + width/2, clip_err, width, label="Mean Error (px)", color="#d9534f", alpha=0.85)

# Annotate
for bar, val in zip(bars1, clip_sr):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold", color="#2d6a2d")

for bar, val in zip(bars2, clip_err):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f"{val}px", ha="center", va="bottom", fontsize=11, fontweight="bold", color="#8b0000")

ax.set_ylabel("Success Rate (%)", fontsize=12, color="#2d6a2d")
ax2.set_ylabel("Mean Pixel Error (px)", fontsize=12, color="#8b0000")
ax.set_xticks(x)
ax.set_xticklabels(clip_models, fontsize=12)
ax.set_title("CLIP Architecture Progression", fontsize=14)
ax.set_ylim(0, 10)
ax2.set_ylim(250, 360)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc="upper left")

# Arrows showing improvement direction
ax.annotate("", xy=(1 - width/2, 2.73), xytext=(0 - width/2, 1.64),
            arrowprops=dict(arrowstyle="->", color="green", lw=1.5))
ax.annotate("", xy=(2 - width/2, 6.01), xytext=(1 - width/2, 2.73),
            arrowprops=dict(arrowstyle="->", color="green", lw=1.5))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "clip_progression.png"), dpi=150)
plt.show()